In [ ]:
import requests
import zipfile
import io
import xml.etree.ElementTree as ET

# 1. 문서에 명시된 기본 정보
BASE_URL = "https://opendart.fss.or.kr/api/corpCode.xml"
API_KEY = "9fe5104553c7e5a643a7bd23c9d6f6b0a6abb817" # 40자리 인증키

def search_corp_details(target_name):
    # 2. 요청 인자(crtfc_key) 설정
    params = {'crtfc_key': API_KEY}
    
    try:
        # 3. GET 방식으로 호출 (응답은 Zip FILE binary)
        response = requests.get(BASE_URL, params=params)
        
        # 응답이 정상인지 확인 (status_code 200)
        if response.status_code != 200:
            return f"접속 실패: {response.status_code}"

        # 4. ZIP 파일 압축 해제 및 XML 파싱
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            # 가이드에 적힌 대로 ZIP 안의 CORPCODE.xml을 읽음
            with z.open('CORPCODE.xml') as f:
                tree = ET.parse(f)
                root = tree.getroot()
                
                # 5. 가이드에 정의된 응답키(list 하위)들 추출
                for item in root.findall('list'):
                    corp_name = item.findtext('corp_name') # 정식명칭
                    
                    if corp_name == target_name:
                        return {
                            "고유번호(corp_code)": item.findtext('corp_code'),
                            "정식명칭(corp_name)": corp_name,
                            "종목코드(stock_code)": item.findtext('stock_code'),
                            "최종변경일자(modify_date)": item.findtext('modify_date')
                        }
        
        return "결과를 찾을 수 없습니다."

    except Exception as e:
        return f"오류 발생: {e}"


{'고유번호(corp_code)': '00365387', '정식명칭(corp_name)': 'AJ네트웍스', '종목코드(stock_code)': '095570', '최종변경일자(modify_date)': '20260327'}


In [37]:

def get_company_details(corp_code):
    # 기업개황 URL
    url = "https://opendart.fss.or.kr/api/company.json"
    
    # 요청 인자
    params = {
        'crtfc_key': API_KEY,
        'corp_code': corp_code
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        # 결과 확인 (status '000'이 정상)
        if data.get('status') == '000':
            print(f"--- {data.get('corp_name')} 상세정보 ---")
            print(f"사업자등록번호: {data.get('bizr_no')}")
            print(f"법인등록번호: {data.get('jurir_no')}")
            print(f"대표자명: {data.get('ceo_nm')}")
            print(f"주소: {data.get('adr')}")
            print(f"홈페이지: {data.get('hm_url')}")
            print(f"업종: {data.get('induty_code')} ({data.get('corp_name_eng')})")
        else:
            print(f"조회 실패: {data.get('message')}")
            
    except Exception as e:
        print(f"오류 발생: {e}")


In [51]:
company_name = "인오션"
dart_detail = search_corp_details(company_name)
dart_code = dart_detail['고유번호(corp_code)']

company_detail = get_company_details(dart_code)
print(company_detail)

TypeError: string indices must be integers, not 'str'

In [45]:
dart_detail

'결과를 찾을 수 없습니다.'